# Filtered Evaluation — RCA > 0.25 Subset

Evaluates all 8 prediction methods on the subset of test pairs where the country already has **RCA > 0.25** in the target product at test year (2015), but has not yet sustained RCA ≥ 1 (which is the label condition).

These are **near-miss pairs** — the country has genuine activity in the product. This is the most economically interesting subset: a model that gets these right is actually identifying which latent capabilities will cross the threshold.

**Filter stats:**  
- Full test set: 127,531 pairs, 14.5% positive rate  
- RCA > 0.25 subset: ~21,041 pairs, ~55% positive rate

Model scores are computed on the **full test set** (models are unaware of the filter). Metrics are then computed on the filtered subset only.

**Metrics:**

| Metric | Definition |
|--------|------------|
| PR-AUC | Area under the Precision-Recall curve |
| AUROC | Area under the ROC curve |
| NDCG@20 | Normalised Discounted Cumulative Gain at rank 20, per country |
| Prec@20 | Precision at rank 20, per country |
| CWR | Complexity-Weighted Recall (top-50% predictions, PCI-weighted) |
| Best F1 | F1 at the threshold that maximises F1 |
| Prec@1000 | Fraction of top-1000 scored pairs that are true positives |
| mAP@10 | Mean Average Precision at rank 10, per country |

In [1]:
import os, sys, pickle, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import precision_recall_curve, auc, roc_auc_score, ndcg_score
from torch_geometric.data import HeteroData
from torch_geometric.nn import SAGEConv, to_hetero
warnings.filterwarnings('ignore')

DATA_DIR     = 'data'
TEST_YEAR    = 2015
TRAIN_CUTOFF = 2012
RCA_THRESH   = 0.25
DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

# ── Raw data ──────────────────────────────────────────────────────────────────
smooth    = pd.read_csv(os.path.join(DATA_DIR, 'M_cpt_smoothed.csv'))
rca_df    = pd.read_csv(os.path.join(DATA_DIR, 'rca_cpt.csv'))
test_lbl  = pd.read_csv(os.path.join(DATA_DIR, 'test_labels.csv'))
train_lbl = pd.read_csv(os.path.join(DATA_DIR, 'train_labels.csv'))

countries = sorted(smooth['country'].unique())
products  = sorted(smooth['product'].unique())
C, P = len(countries), len(products)
c_idx = {c: i for i, c in enumerate(countries)}
p_idx = {p: i for i, p in enumerate(products)}

def build_M(year):
    M = np.zeros((C, P), dtype=np.float32)
    yr = smooth[smooth['year'] == year]
    M[yr['country'].map(c_idx).values, yr['product'].map(p_idx).values] = 1.0
    return M

M_t    = build_M(TEST_YEAR)

# ── Build the filtered test set (RCA > 0.25 at test year) ─────────────────────
rca_test = rca_df[rca_df['year'] == TEST_YEAR][['country', 'product', 'rca']]
test_merged = test_lbl.merge(rca_test, on=['country', 'product'], how='left')
test_merged['rca'] = test_merged['rca'].fillna(0.0)
filt_lbl = test_merged[test_merged['rca'] > RCA_THRESH].copy().reset_index(drop=True)

print(f'Full test set:      {len(test_lbl):,} pairs  ({test_lbl["label"].sum():,} positives, {test_lbl["label"].mean()*100:.1f}% rate)')
print(f'RCA>{RCA_THRESH} filtered: {len(filt_lbl):,} pairs  ({filt_lbl["label"].sum():,} positives, {filt_lbl["label"].mean()*100:.1f}% rate)')
print(f'Countries in filtered set: {filt_lbl["country"].nunique()}')

y_true_full = test_lbl['label'].values
y_true_filt = filt_lbl['label'].values

# ── Proximity matrix (train years only) ───────────────────────────────────────
print('\nBuilding proximity matrix...')
co_exp = np.zeros((P, P), dtype=np.float32)
any_exp = np.zeros((P, P), dtype=np.float32)
for yr in sorted(y for y in smooth['year'].unique() if y <= TRAIN_CUTOFF):
    M = build_M(yr)
    co = M.T @ M
    ex = M.sum(axis=0)
    co_exp  += co
    any_exp += ex[:, None] + ex[None, :] - co
phi = np.where(any_exp > 0, co_exp / (any_exp + 1e-9), 0.0)
np.fill_diagonal(phi, 0.0)
phi_row_sum = phi.sum(axis=1)

# ── ECI ───────────────────────────────────────────────────────────────────────
kc = M_t.sum(axis=1); kp = M_t.sum(axis=0)
kc_safe = np.where(kc > 0, kc, 1.0); kp_safe = np.where(kp > 0, kp, 1.0)
kc_n, kp_n = kc.astype(float), kp.astype(float)
for _ in range(20):
    kc_n = (1.0 / kc_safe) * (M_t   @ kp_n)
    kp_n = (1.0 / kp_safe) * (M_t.T @ kc_n)
eci = (kc_n - kc_n.mean()) / (kc_n.std() + 1e-9)

# ── PCI proxy for CWR ─────────────────────────────────────────────────────────
rca_ref  = rca_df[rca_df['year'] == 2010]
ubiq     = rca_ref.groupby('product')['rca'].apply(lambda x: (x >= 1).sum())
max_ubiq = ubiq.max()
pci_dict = {int(p): float(-u / max_ubiq) for p, u in ubiq.items()}
filt_lbl['pci'] = filt_lbl['product'].map(pci_dict).fillna(0.0)
min_pci = filt_lbl['pci'].min()
filt_lbl['w'] = filt_lbl['pci'] - min_pci

def minmax(x): return (x - x.min()) / (x.max() - x.min() + 1e-9)

print('Setup done.')

Device: cuda
Full test set:      127,531 pairs  (18,477 positives, 14.5% rate)
RCA>0.25 filtered: 21,041 pairs  (11,648 positives, 55.4% rate)
Countries in filtered set: 225

Building proximity matrix...
Setup done.


## Metric Definitions

In [2]:
def compute_all_metrics(scores_full, filt_df, skip_map10=False):
    """
    scores_full : score array aligned with the FULL test_lbl (127K rows)
    filt_df     : the RCA>0.25 filtered DataFrame with 'label', 'country', 'product', 'w' columns

    Scores are re-aligned to filt_df by merging, then all metrics are computed
    on the filtered subset only.
    """
    # Align full scores to filtered subset
    tmp = test_lbl.copy()
    tmp['score'] = scores_full
    df = filt_df.merge(tmp[['country', 'product', 'score']],
                       on=['country', 'product'], how='left').fillna(0)
    scores = df['score'].values
    labels = df['label'].values

    # PR-AUC
    p, r, thresholds = precision_recall_curve(labels, scores)
    pr_auc = auc(r, p)

    # AUROC
    auroc = roc_auc_score(labels, scores) if 0 < labels.mean() < 1 else 0.0

    # Best F1
    denom = p + r
    f1_all = np.where(denom > 0, 2 * p * r / denom, 0.0)
    best_f1 = float(f1_all.max())

    # Prec@1000  (or @N if fewer than 1000 pairs)
    K1000 = min(1000, len(scores))
    topk  = np.argsort(scores)[::-1][:K1000]
    p1k   = float(labels[topk].sum()) / K1000

    # NDCG@20, Prec@20, mAP@10 — all per country
    ndcg_vals, prec20_vals, ap10_vals = [], [], []
    for _, grp in df.groupby('country'):
        if grp['label'].sum() == 0:
            continue
        yt = grp['label'].values
        ys = grp['score'].values

        try:
            ndcg_vals.append(ndcg_score([yt], [ys], k=20))
        except Exception:
            pass
        prec20_vals.append(
            grp.sort_values('score', ascending=False).head(20)['label'].mean())

        if not skip_map10:
            n_pos  = int(grp['label'].sum())
            top10  = grp.sort_values('score', ascending=False).head(10)['label'].values
            cumtp  = np.cumsum(top10)
            prec_k = cumtp / np.arange(1, len(top10) + 1)
            ap     = (prec_k * top10).sum() / min(n_pos, 10)
            ap10_vals.append(ap)

    ndcg20 = float(np.nanmean(ndcg_vals))  if ndcg_vals  else 0.0
    prec20 = float(np.nanmean(prec20_vals)) if prec20_vals else 0.0
    map10  = float(np.nanmean(ap10_vals))   if ap10_vals  else float('nan')

    # CWR
    df['score_pct'] = df['score'].rank(pct=True)
    tot_w = df.loc[df['label'] == 1, 'w'].sum()
    hit_w = df.loc[(df['label'] == 1) & (df['score_pct'] >= 0.5), 'w'].sum()
    cwr   = float(hit_w / tot_w) if tot_w > 0 else 0.0

    return {
        'PR-AUC':   round(pr_auc,  4),
        'AUROC':    round(auroc,   4),
        'NDCG@20':  round(ndcg20,  4),
        'Prec@20':  round(prec20,  4),
        'CWR':      round(cwr,     4),
        'Best F1':  round(best_f1, 4),
        f'P@{K1000}': round(p1k,  4),
        'mAP@10':   'N/A' if skip_map10 else round(map10, 4),
    }

ALL_RESULTS = {}
NO_MAP_AT10 = set()

def evaluate(name, scores_full, skip_map10=False):
    if skip_map10:
        NO_MAP_AT10.add(name)
    res = compute_all_metrics(scores_full, filt_lbl, skip_map10=skip_map10)
    ALL_RESULTS[name] = res
    m10s = res['mAP@10'] if skip_map10 else f'{res["mAP@10"]:.4f}'
    p1k_key = [k for k in res if k.startswith('P@')][0]
    print(f'{name:<26}  PR-AUC={res["PR-AUC"]:.4f}  NDCG@20={res["NDCG@20"]:.4f}  '
          f'Prec@20={res["Prec@20"]:.4f}  CWR={res["CWR"]:.4f}  '
          f'BestF1={res["Best F1"]:.4f}  {p1k_key}={res[p1k_key]:.4f}  mAP@10={m10s}')
    return res

print('Metric functions ready.')

Metric functions ready.


## Method 1 — RCA Persistence

In [3]:
history_years = [TEST_YEAR - 2, TEST_YEAR - 1, TEST_YEAR]
rca_hist = rca_df[rca_df['year'].isin(history_years)][['country', 'product', 'year', 'rca']]
rca_hist = rca_hist.merge(test_lbl[['country', 'product']], on=['country', 'product'])
rca_wide = rca_hist.pivot_table(index=['country', 'product'], columns='year', values='rca', fill_value=0)
for yr in history_years:
    if yr not in rca_wide.columns:
        rca_wide[yr] = 0
rca_wide['score'] = (rca_wide[history_years] >= 1).mean(axis=1)
persist_df = test_lbl.merge(rca_wide[['score']], on=['country', 'product'], how='left').fillna(0)
evaluate('RCA Persistence', persist_df['score'].values)

RCA Persistence             PR-AUC=0.7396  NDCG@20=0.7071  Prec@20=0.5919  CWR=0.4717  BestF1=0.7127  P@1000=0.7030  mAP@10=0.5387


{'PR-AUC': 0.7396,
 'AUROC': 0.6193,
 'NDCG@20': 0.7071,
 'Prec@20': 0.5919,
 'CWR': 0.4717,
 'Best F1': 0.7127,
 'P@1000': 0.703,
 'mAP@10': 0.5387}

## Method 2 — Product Space Density

In [4]:
ci_full = test_lbl['country'].map(c_idx).values
pi_full = test_lbl['product'].map(p_idx).values
dens_mat    = (M_t @ phi) / (phi_row_sum[None, :] + 1e-9)
dens_scores = dens_mat[ci_full, pi_full]
evaluate('Density', dens_scores)

Density                     PR-AUC=0.5808  NDCG@20=0.7375  Prec@20=0.5944  CWR=0.5352  BestF1=0.7157  P@1000=0.6190  mAP@10=0.5951


{'PR-AUC': 0.5808,
 'AUROC': 0.5422,
 'NDCG@20': 0.7375,
 'Prec@20': 0.5944,
 'CWR': 0.5352,
 'Best F1': 0.7157,
 'P@1000': 0.619,
 'mAP@10': 0.5951}

## Method 3 — ECI

ECI is per-country only — all products within a country share the same score, so **mAP@10 is N/A**.

In [5]:
evaluate('ECI', eci[ci_full], skip_map10=True)

ECI                         PR-AUC=0.5408  NDCG@20=0.5936  Prec@20=0.4924  CWR=0.4875  BestF1=0.7127  P@1000=0.5230  mAP@10=N/A


{'PR-AUC': 0.5408,
 'AUROC': 0.4814,
 'NDCG@20': 0.5936,
 'Prec@20': 0.4924,
 'CWR': 0.4875,
 'Best F1': 0.7127,
 'P@1000': 0.523,
 'mAP@10': 'N/A'}

## Method 4 — ECI + Density

In [6]:
evaluate('ECI + Density', minmax(eci[ci_full]) + minmax(dens_mat[ci_full, pi_full]))

ECI + Density               PR-AUC=0.5808  NDCG@20=0.7375  Prec@20=0.5944  CWR=0.5352  BestF1=0.7157  P@1000=0.6190  mAP@10=0.5951


{'PR-AUC': 0.5808,
 'AUROC': 0.5422,
 'NDCG@20': 0.7375,
 'Prec@20': 0.5944,
 'CWR': 0.5352,
 'Best F1': 0.7157,
 'P@1000': 0.619,
 'mAP@10': 0.5951}

## Method 5 — KNN on LLM Embeddings

In [7]:
EMB_PATH = os.path.join(DATA_DIR, 'product_llm_embeddings.pt')
emb = torch.load(EMB_PATH, weights_only=False, map_location='cpu').numpy()
print(f'Embeddings: {emb.shape}')

country_basket = {}
for c in test_lbl['country'].unique():
    ci = c_idx.get(c, -1)
    if ci < 0:
        continue
    exported = np.where(M_t[ci] == 1)[0]
    if len(exported) == 0:
        country_basket[c] = np.zeros(emb.shape[1])
    else:
        basket = emb[exported].mean(axis=0)
        country_basket[c] = basket / (np.linalg.norm(basket) + 1e-9)

knn_scores_full = np.array([
    float(emb[p_idx[p]] @ country_basket[c])
    if c in country_basket and p in p_idx else 0.0
    for c, p in zip(test_lbl['country'].values, test_lbl['product'].values)
], dtype=np.float32)

evaluate('KNN (LLM embeddings)', knn_scores_full)

Embeddings: (5018, 768)
KNN (LLM embeddings)        PR-AUC=0.5856  NDCG@20=0.6280  Prec@20=0.5166  CWR=0.5049  BestF1=0.7127  P@1000=0.6700  mAP@10=0.4450


{'PR-AUC': 0.5856,
 'AUROC': 0.5294,
 'NDCG@20': 0.628,
 'Prec@20': 0.5166,
 'CWR': 0.5049,
 'Best F1': 0.7127,
 'P@1000': 0.67,
 'mAP@10': 0.445}

## GNN Architecture & Checkpoint Loader

In [8]:
CKPT_DIR = os.path.join(DATA_DIR, 'models', 'gnn', 'checkpoints')

edge_idx_raw   = torch.load(os.path.join(DATA_DIR, 'edge_index_by_year.pt'), weights_only=False)
edge_idx_by_yr = {k: v.long() for k, v in edge_idx_raw.items()}
p_x_by_yr      = torch.load(os.path.join(DATA_DIR, 'product_x_by_year.pt'),  weights_only=False)
c_x_11feat     = torch.load(os.path.join(DATA_DIR, 'country_x_by_year.pt'),  weights_only=False)
cap_ei         = torch.load(os.path.join(DATA_DIR, 'capability_edge_index.pt'), weights_only=False).long()

with open(os.path.join(DATA_DIR, 'country_mapping.pkl'), 'rb') as f: c_map = pickle.load(f)
with open(os.path.join(DATA_DIR, 'product_mapping.pkl'), 'rb') as f: p_map = pickle.load(f)

c_feat_df = pd.read_csv(os.path.join(DATA_DIR, 'country_features.csv'))
BACI_COLS = ['log_export', 'n_products', 'avg_rca', 'max_rca']
c_x_4feat = {}
for yr in sorted(c_feat_df['year'].unique()):
    yd = c_feat_df[c_feat_df['year'] == yr].copy()
    yd['idx'] = yd['country'].map(c_map['to_idx'])
    yd = yd.dropna(subset=['idx']).sort_values('idx')
    c_x_4feat[int(yr)] = torch.tensor(yd[BACI_COLS].values, dtype=torch.float32)

class _HomoGNN(nn.Module):
    def __init__(self, hidden, drop=0.3):
        super().__init__()
        self.c1 = SAGEConv(hidden, hidden); self.c2 = SAGEConv(hidden, hidden); self.drop = drop
    def forward(self, x, edge_index):
        x = F.dropout(self.c1(x, edge_index).relu(), p=self.drop, training=self.training)
        return self.c2(x, edge_index)

class BipartiteEncoder(nn.Module):
    def __init__(self, c_in, hidden, meta):
        super().__init__()
        self.country_lin = nn.Linear(c_in, hidden)
        self.product_lin = nn.Linear(3, hidden)
        self.gnn = to_hetero(_HomoGNN(hidden), meta)
    def forward(self, x_dict, ei_dict):
        return self.gnn({'country': self.country_lin(x_dict['country']),
                         'product': self.product_lin(x_dict['product'])}, ei_dict)

class TemporalGNN(nn.Module):
    def __init__(self, enc, hidden):
        super().__init__()
        self.enc   = enc
        self.gru_c = nn.GRU(hidden, hidden)
        self.gru_p = nn.GRU(hidden, hidden)
    def forward(self, snaps):
        cs, ps = [], []
        for s in snaps:
            z = self.enc(s.x_dict, s.edge_index_dict)
            cs.append(z['country']); ps.append(z['product'])
        z_c, _ = self.gru_c(torch.stack(cs))
        z_p, _ = self.gru_p(torch.stack(ps))
        return {'country': z_c[-1], 'product': z_p[-1]}

class LinkPredictor(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(hidden * 2, hidden), nn.ReLU(), nn.Dropout(0.2), nn.Linear(hidden, 1))
    def forward(self, zc, zp, ei):
        return self.mlp(torch.cat([zc[ei[0]], zp[ei[1]]], -1)).view(-1)

@torch.no_grad()
def gnn_scores_full(ckpt_path, c_x, with_cap=False):
    """Return score array aligned with test_lbl (full 127K rows)."""
    ckpt = torch.load(ckpt_path, weights_only=False, map_location=DEVICE)
    enc  = BipartiteEncoder(ckpt['c_in'], ckpt['hidden'], ckpt['meta']).to(DEVICE)
    mdl  = TemporalGNN(enc, ckpt['hidden']).to(DEVICE)
    pred = LinkPredictor(ckpt['hidden']).to(DEVICE)
    mdl.load_state_dict({k: v.to(DEVICE) for k, v in ckpt['mdl_state'].items()})
    pred.load_state_dict({k: v.to(DEVICE) for k, v in ckpt['pred_state'].items()})
    mdl.eval(); pred.eval()

    snaps = []
    for y in range(TEST_YEAR - 4, TEST_YEAR + 1):
        d = HeteroData()
        d['country'].x = c_x[y].to(DEVICE)
        d['product'].x = p_x_by_yr[y].to(DEVICE)
        ei = edge_idx_by_yr[y].long().to(DEVICE)
        d['country', 'exports',     'product'].edge_index = ei
        d['product', 'rev_exports', 'country'].edge_index = ei.flip(0)
        if with_cap:
            d['product', 'capability', 'product'].edge_index = cap_ei.to(DEVICE)
        snaps.append(d)

    ci_s = test_lbl['country'].map(c_map['to_idx'])
    pi_s = test_lbl['product'].map(p_map['to_idx'])
    ok   = ci_s.notna() & pi_s.notna()
    ci_v = ci_s[ok].astype(int).values
    pi_v = pi_s[ok].astype(int).values
    ei_t = torch.tensor([ci_v, pi_v], dtype=torch.long).to(DEVICE)

    z   = mdl(snaps)
    raw = torch.sigmoid(pred(z['country'], z['product'], ei_t)).cpu().numpy()

    gnn_df = pd.DataFrame({'country': test_lbl.loc[ok, 'country'].values,
                           'product': test_lbl.loc[ok, 'product'].values,
                           'score':   raw})
    return test_lbl.merge(gnn_df[['country', 'product', 'score']],
                          on=['country', 'product'], how='left').fillna(0)['score'].values

print('GNN helpers loaded.')

GNN helpers loaded.


## Method 6 — GNN-4F

In [9]:
CKPT_4F = os.path.join(CKPT_DIR, 'gnn_4f.pt')
print('Loading GNN-4F...')
evaluate('GNN-4F', gnn_scores_full(CKPT_4F, c_x_4feat, with_cap=False))

Loading GNN-4F...
GNN-4F                      PR-AUC=0.6211  NDCG@20=0.7038  Prec@20=0.5734  CWR=0.5322  BestF1=0.7181  P@1000=0.6800  mAP@10=0.5505


{'PR-AUC': 0.6211,
 'AUROC': 0.593,
 'NDCG@20': 0.7038,
 'Prec@20': 0.5734,
 'CWR': 0.5322,
 'Best F1': 0.7181,
 'P@1000': 0.68,
 'mAP@10': 0.5505}

## Method 7 — GNN-11F

In [10]:
CKPT_11F = os.path.join(CKPT_DIR, 'gnn_11f.pt')
print('Loading GNN-11F...')
evaluate('GNN-11F (BACI+WDI)', gnn_scores_full(CKPT_11F, c_x_11feat, with_cap=False))

Loading GNN-11F...
GNN-11F (BACI+WDI)          PR-AUC=0.6515  NDCG@20=0.7143  Prec@20=0.5789  CWR=0.5467  BestF1=0.7209  P@1000=0.7300  mAP@10=0.5675


{'PR-AUC': 0.6515,
 'AUROC': 0.6199,
 'NDCG@20': 0.7143,
 'Prec@20': 0.5789,
 'CWR': 0.5467,
 'Best F1': 0.7209,
 'P@1000': 0.73,
 'mAP@10': 0.5675}

## Method 8 — GNN-11F+LLM

In [11]:
CKPT_LLM = os.path.join(CKPT_DIR, 'gnn_11f_llm.pt')
print('Loading GNN-11F+LLM...')
evaluate('GNN-11F+LLM', gnn_scores_full(CKPT_LLM, c_x_11feat, with_cap=True))

Loading GNN-11F+LLM...
GNN-11F+LLM                 PR-AUC=0.6590  NDCG@20=0.7182  Prec@20=0.5801  CWR=0.5521  BestF1=0.7196  P@1000=0.7580  mAP@10=0.5674


{'PR-AUC': 0.659,
 'AUROC': 0.6245,
 'NDCG@20': 0.7182,
 'Prec@20': 0.5801,
 'CWR': 0.5521,
 'Best F1': 0.7196,
 'P@1000': 0.758,
 'mAP@10': 0.5674}

## Results Table

In [12]:
METHOD_ORDER = [
    'RCA Persistence', 'Density', 'ECI', 'ECI + Density',
    'KNN (LLM embeddings)', 'GNN-4F', 'GNN-11F (BACI+WDI)', 'GNN-11F+LLM'
]
methods_run = [m for m in METHOD_ORDER if m in ALL_RESULTS]

# Detect the P@K key (might be P@1000 or P@N for smaller subsets)
p1k_key = next((k for k in ALL_RESULTS[methods_run[0]] if k.startswith('P@')), 'P@1000')

METRICS = ['PR-AUC', 'AUROC', 'NDCG@20', 'Prec@20', 'CWR', 'Best F1', p1k_key, 'mAP@10']

print(f'Filtered evaluation on RCA > {RCA_THRESH} pairs '
      f'({len(filt_lbl):,} pairs, {filt_lbl["label"].mean()*100:.1f}% positive rate)')
print('=' * 100)
hdr = f'  {"Method":<26}' + ''.join(f'{m:>10}' for m in METRICS)
print(hdr)
print('-' * 100)
rows = []
for m in methods_run:
    r   = ALL_RESULTS[m]
    gnn = ' <--' if 'GNN' in m else ''
    vals = []
    row_dict = {'Method': m}
    for met in METRICS:
        v = r.get(met, r.get(p1k_key if met == p1k_key else met, 'N/A'))
        row_dict[met] = v
        if v == 'N/A':
            vals.append(f'{'N/A':>10}')
        else:
            vals.append(f'{v:>10.4f}')
    print(f'  {m:<26}' + ''.join(vals) + gnn)
    rows.append(row_dict)
print('=' * 100)
print('N/A = ECI has no product-level ranking signal.')
print(f'Note: {p1k_key} uses top-{p1k_key[2:]} of the {len(filt_lbl):,} filtered pairs scored globally.')

df_res = pd.DataFrame(rows).set_index('Method')
df_res.to_csv(os.path.join(DATA_DIR, 'filtered_metrics_results.csv'))
print('\nSaved -> data/filtered_metrics_results.csv')

Filtered evaluation on RCA > 0.25 pairs (21,041 pairs, 55.4% positive rate)
  Method                        PR-AUC     AUROC   NDCG@20   Prec@20       CWR   Best F1    P@1000    mAP@10
----------------------------------------------------------------------------------------------------
  RCA Persistence               0.7396    0.6193    0.7071    0.5919    0.4717    0.7127    0.7030    0.5387
  Density                       0.5808    0.5422    0.7375    0.5944    0.5352    0.7157    0.6190    0.5951
  ECI                           0.5408    0.4814    0.5936    0.4924    0.4875    0.7127    0.5230       N/A
  ECI + Density                 0.5808    0.5422    0.7375    0.5944    0.5352    0.7157    0.6190    0.5951
  KNN (LLM embeddings)          0.5856    0.5294    0.6280    0.5166    0.5049    0.7127    0.6700    0.4450
  GNN-4F                        0.6211    0.5930    0.7038    0.5734    0.5322    0.7181    0.6800    0.5505 <--
  GNN-11F (BACI+WDI)            0.6515    0.6199    0.71